# MVRV Trailing Stop Grid Search

**Goal:** Find optimal combination of:
- MVRV trigger level (when to activate trailing stop)
- Trailing stop percentage (how much drawdown before exit)

Current best: MVRV > 2.25 → 20% trail = 62% beat rate

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("MVRV Grid Search 🔍")

In [ ]:
# Load data
DATA_DIR = Path("../data/raw")

sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")

df = sopr.join(sopr_sth, how='inner').join(price, how='inner').join(mvrv, how='inner')
df = df.sort_index()
df = df[df.index >= '2018-12-15']

close = df['price']
print(f"Data: {len(df)} rows")

In [ ]:
# Entry signal
both_below_1 = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
entries = both_below_1 & ~both_below_1.shift(1).fillna(False)
print(f"Entry signals: {entries.sum()}")

In [ ]:
def backtest_mvrv_trailing(
    df, entries,
    mvrv_trigger=2.0,
    trailing_pct=0.15,
    stop_loss=0.20,
    max_hold_days=365
):
    """MVRV triggers trailing stop."""
    trades = []
    entry_indices = entries[entries].index.tolist()
    close = df['price']
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = df.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        peak_price = entry_price
        trailing_active = False
        
        exit_date = None
        exit_price = None
        exit_reason = None
        
        for j in range(entry_idx + 1, len(df)):
            current_date = df.index[j]
            current_price = close.iloc[j]
            current_mvrv = df['mvrv'].iloc[j]
            days_held = j - entry_idx
            
            if current_price > peak_price:
                peak_price = current_price
            
            pnl = (current_price - entry_price) / entry_price
            
            # Activate trailing when MVRV hits trigger
            if not trailing_active and current_mvrv >= mvrv_trigger:
                trailing_active = True
            
            # Check trailing stop if active
            if trailing_active:
                trail_stop = peak_price * (1 - trailing_pct)
                if current_price <= trail_stop:
                    exit_date = current_date
                    exit_price = trail_stop
                    exit_reason = 'mvrv_trail'
                    break
            
            # Initial stop loss
            if not trailing_active and stop_loss and pnl <= -stop_loss:
                exit_date = current_date
                exit_price = entry_price * (1 - stop_loss)
                exit_reason = 'stop_loss'
                break
            
            # Max hold
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'max_hold'
                break
        
        if exit_date is None:
            exit_date = df.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        pnl = (exit_price - entry_price) / entry_price
        trades.append({
            'entry_date': entry_date,
            'exit_date': exit_date,
            'pnl_pct': pnl,
            'exit_reason': exit_reason,
            'trail_activated': trailing_active
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

In [ ]:
def walk_forward(df, entries, mvrv_trigger, trailing_pct, stop_loss=0.20):
    """Walk-forward validation."""
    results = []
    close = df['price']
    
    train_days = 365
    test_days = 90
    step_days = 90
    
    total_days = len(df)
    n_folds = (total_days - train_days) // step_days
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        test_df = df.iloc[test_start:test_end]
        test_entries = entries.iloc[test_start:test_end]
        test_close = close.iloc[test_start:test_end]
        
        trades = backtest_mvrv_trailing(
            test_df, test_entries,
            mvrv_trigger=mvrv_trigger,
            trailing_pct=trailing_pct,
            stop_loss=stop_loss
        )
        
        strat_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        results.append({
            'strat_return': strat_return,
            'hold_return': hold_return,
            'beat_hold': strat_return > hold_return
        })
    
    wf_df = pd.DataFrame(results)
    return wf_df['beat_hold'].mean(), (wf_df['strat_return'] - wf_df['hold_return']).mean()

---
## Grid Search

In [ ]:
# Define grid
mvrv_triggers = [1.5, 1.75, 2.0, 2.25, 2.5, 2.75, 3.0]
trailing_pcts = [0.10, 0.15, 0.20, 0.25, 0.30]

print(f"Testing {len(mvrv_triggers)} MVRV triggers x {len(trailing_pcts)} trail % = {len(mvrv_triggers)*len(trailing_pcts)} combinations")
print("This may take a minute...\n")

In [ ]:
# Run grid search
grid_results = []

for trigger in mvrv_triggers:
    for trail in trailing_pcts:
        beat_rate, avg_excess = walk_forward(df, entries, trigger, trail)
        
        # Also get in-sample stats
        trades = backtest_mvrv_trailing(df, entries, mvrv_trigger=trigger, trailing_pct=trail)
        total_return = (1 + trades['pnl_pct']).prod() - 1
        trail_exits = (trades['exit_reason'] == 'mvrv_trail').sum()
        
        grid_results.append({
            'mvrv_trigger': trigger,
            'trail_pct': trail,
            'beat_rate': beat_rate,
            'avg_excess': avg_excess,
            'in_sample_return': total_return,
            'trail_exits': trail_exits,
            'total_trades': len(trades)
        })

grid_df = pd.DataFrame(grid_results)
print("Grid search complete!")

In [ ]:
# Display results table
print("GRID SEARCH RESULTS")
print("="*100)
print(f"{'MVRV Trigger':<14} {'Trail %':<10} {'Beat Rate':>12} {'Avg Excess':>12} {'Trail Exits':>12} {'In-Sample':>12}")
print("-"*100)

# Sort by beat rate
sorted_df = grid_df.sort_values('beat_rate', ascending=False)

for _, row in sorted_df.iterrows():
    print(f"MVRV > {row['mvrv_trigger']:<6} {row['trail_pct']*100:>6.0f}% "
          f"{row['beat_rate']*100:>11.0f}% {row['avg_excess']*100:>+11.1f}% "
          f"{row['trail_exits']:>12} {row['in_sample_return']*100:>11.0f}%")

In [ ]:
# Create heatmap of beat rates
pivot_beat = grid_df.pivot(index='mvrv_trigger', columns='trail_pct', values='beat_rate')

fig = go.Figure(data=go.Heatmap(
    z=pivot_beat.values * 100,
    x=[f"{int(x*100)}%" for x in pivot_beat.columns],
    y=[f"MVRV > {x}" for x in pivot_beat.index],
    colorscale='RdYlGn',
    zmid=54,  # Baseline
    text=[[f"{val:.0f}%" for val in row] for row in pivot_beat.values * 100],
    texttemplate="%{text}",
    textfont={"size": 12},
    hovertemplate="MVRV: %{y}<br>Trail: %{x}<br>Beat Rate: %{z:.0f}%<extra></extra>"
))

fig.update_layout(
    title='Walk-Forward Beat Rate by MVRV Trigger & Trail %<br><sup>Green = Better than baseline (54%), Red = Worse</sup>',
    xaxis_title='Trailing Stop %',
    yaxis_title='MVRV Trigger Level',
    height=500
)
fig.show()

In [ ]:
# Create heatmap of avg excess
pivot_excess = grid_df.pivot(index='mvrv_trigger', columns='trail_pct', values='avg_excess')

fig = go.Figure(data=go.Heatmap(
    z=pivot_excess.values * 100,
    x=[f"{int(x*100)}%" for x in pivot_excess.columns],
    y=[f"MVRV > {x}" for x in pivot_excess.index],
    colorscale='RdYlGn',
    zmid=0,
    text=[[f"{val:+.1f}%" for val in row] for row in pivot_excess.values * 100],
    texttemplate="%{text}",
    textfont={"size": 12},
))

fig.update_layout(
    title='Average Excess Return by MVRV Trigger & Trail %<br><sup>Green = Positive, Red = Negative</sup>',
    xaxis_title='Trailing Stop %',
    yaxis_title='MVRV Trigger Level',
    height=500
)
fig.show()

In [ ]:
# Find top 10 configurations
print("\n\nTOP 10 CONFIGURATIONS BY BEAT RATE")
print("="*90)
print(f"{'Rank':<6} {'Config':<25} {'Beat Rate':>12} {'Avg Excess':>12} {'Trail Exits':>12}")
print("-"*90)

top10 = sorted_df.head(10)
for rank, (_, row) in enumerate(top10.iterrows(), 1):
    config = f"MVRV>{row['mvrv_trigger']} → {row['trail_pct']*100:.0f}%"
    print(f"{rank:<6} {config:<25} {row['beat_rate']*100:>11.0f}% {row['avg_excess']*100:>+11.1f}% {row['trail_exits']:>12}")

In [ ]:
# Analyze patterns
print("\n\nPATTERN ANALYSIS")
print("="*60)

# Average beat rate by MVRV trigger
print("\nAverage Beat Rate by MVRV Trigger:")
by_trigger = grid_df.groupby('mvrv_trigger')['beat_rate'].mean().sort_values(ascending=False)
for trigger, rate in by_trigger.items():
    print(f"  MVRV > {trigger}: {rate*100:.1f}%")

# Average beat rate by trailing %
print("\nAverage Beat Rate by Trail %:")
by_trail = grid_df.groupby('trail_pct')['beat_rate'].mean().sort_values(ascending=False)
for trail, rate in by_trail.items():
    print(f"  {trail*100:.0f}% trail: {rate*100:.1f}%")

In [ ]:
# Line plots by trigger level
fig = go.Figure()

for trigger in mvrv_triggers:
    subset = grid_df[grid_df['mvrv_trigger'] == trigger]
    fig.add_trace(go.Scatter(
        x=subset['trail_pct'] * 100,
        y=subset['beat_rate'] * 100,
        mode='lines+markers',
        name=f'MVRV > {trigger}'
    ))

fig.add_hline(y=54, line_dash='dash', line_color='gray', annotation_text='Baseline 54%')

fig.update_layout(
    title='Beat Rate by Trail % (for each MVRV Trigger)',
    xaxis_title='Trailing Stop %',
    yaxis_title='Beat Rate %',
    height=500
)
fig.show()

---
## Best Configuration Deep Dive

In [ ]:
# Get best configuration
best = sorted_df.iloc[0]
print(f"\n🏆 BEST CONFIGURATION")
print(f"   MVRV Trigger: > {best['mvrv_trigger']}")
print(f"   Trailing Stop: {best['trail_pct']*100:.0f}%")
print(f"   Beat Rate: {best['beat_rate']*100:.0f}%")
print(f"   Avg Excess: {best['avg_excess']*100:+.1f}%")

In [ ]:
# Get trades for best config
best_trades = backtest_mvrv_trailing(
    df, entries,
    mvrv_trigger=best['mvrv_trigger'],
    trailing_pct=best['trail_pct']
)

print(f"\nTRADE DETAILS")
print("="*100)

display = best_trades.copy()
display['entry_date'] = pd.to_datetime(display['entry_date']).dt.strftime('%Y-%m-%d')
display['exit_date'] = pd.to_datetime(display['exit_date']).dt.strftime('%Y-%m-%d')
display['pnl_pct'] = (display['pnl_pct'] * 100).round(1)

print(display.to_string(index=False))

print(f"\nExit Breakdown:")
print(best_trades['exit_reason'].value_counts())

In [ ]:
# Compare top 3 to understand robustness
print("\n\nROBUSTNESS CHECK: Top 3 Configs")
print("="*80)

top3 = sorted_df.head(3)
for i, (_, row) in enumerate(top3.iterrows(), 1):
    trades = backtest_mvrv_trailing(df, entries, row['mvrv_trigger'], row['trail_pct'])
    
    print(f"\n#{i}: MVRV > {row['mvrv_trigger']} → {row['trail_pct']*100:.0f}% trail")
    print(f"    Beat Rate: {row['beat_rate']*100:.0f}%")
    print(f"    Total Return (in-sample): {row['in_sample_return']*100:.0f}%")
    print(f"    Exit Breakdown: {trades['exit_reason'].value_counts().to_dict()}")

---
## Summary

In [ ]:
print("\n" + "="*80)
print("GRID SEARCH SUMMARY")
print("="*80)

print(f"\n📊 TESTED: {len(grid_results)} combinations")
print(f"   MVRV triggers: {mvrv_triggers}")
print(f"   Trail %: {[f'{x*100:.0f}%' for x in trailing_pcts]}")

print(f"\n🏆 BEST: MVRV > {best['mvrv_trigger']} → {best['trail_pct']*100:.0f}% trail")
print(f"   Beat Rate: {best['beat_rate']*100:.0f}%")
print(f"   Avg Excess: {best['avg_excess']*100:+.1f}%")

# How many configs beat baseline?
above_baseline = (grid_df['beat_rate'] > 0.54).sum()
print(f"\n📈 {above_baseline}/{len(grid_df)} configs beat baseline (54%)")

# Best MVRV trigger on average
best_trigger = by_trigger.idxmax()
print(f"\n💡 Best MVRV trigger on average: > {best_trigger}")

# Best trail % on average  
best_trail = by_trail.idxmax()
print(f"💡 Best trail % on average: {best_trail*100:.0f}%")

print("\n" + "="*80)

In [ ]:
# Save results
import json

results = {
    'grid_search': {
        'mvrv_triggers': mvrv_triggers,
        'trailing_pcts': trailing_pcts,
        'total_combinations': len(grid_results)
    },
    'best_config': {
        'mvrv_trigger': float(best['mvrv_trigger']),
        'trail_pct': float(best['trail_pct']),
        'beat_rate': float(best['beat_rate']),
        'avg_excess': float(best['avg_excess'])
    },
    'all_results': grid_df.to_dict('records'),
    'pattern_analysis': {
        'by_trigger': by_trigger.to_dict(),
        'by_trail': {f"{k*100:.0f}%": v for k, v in by_trail.to_dict().items()}
    }
}

with open('../data/mvrv_grid_search_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Saved to ../data/mvrv_grid_search_results.json")